In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: d:\Ecommerce-purchase-prediction-DeepLearning


In [2]:
import json
import pandas as pd

from common.config import (
    SEED,
    RAW_EVENTS_PATH,
    PROCESSED_DIR,
    ALLOWED_INPUT_EVENTS,
)

from common.cleaning import load_and_clean_events
from common.sessionization import create_sessions
from common.labeling import create_labels_and_model_events
from common.split import chronological_split
from common.utils import set_seed

In [3]:
set_seed(SEED)

raw_path = PROJECT_ROOT / RAW_EVENTS_PATH
processed_dir = PROJECT_ROOT / PROCESSED_DIR

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Raw data:", raw_path)
print("Processed directory:", processed_dir)
print("Seed:", SEED)

Raw data: d:\Ecommerce-purchase-prediction-DeepLearning\data\raw\events.csv
Processed directory: d:\Ecommerce-purchase-prediction-DeepLearning\data\processed
Seed: 42


In [4]:
clean_events = load_and_clean_events(
    raw_path
)

print("Clean event rows:", len(clean_events))

display(clean_events.head())

Clean event rows: 2755641


,timestamp,datetime,visitorid,event,itemid,transactionid
0,1442004589439,2015-09-11 20:49:49.439,0,view,285930,NaN
1,1442004759591,2015-09-11 20:52:39.591,0,view,357564,NaN
2,1442004917175,2015-09-11 20:55:17.175,0,view,67045,NaN
3,1439487966444,2015-08-13 17:46:06.444,1,view,72028,NaN
4,1438969904567,2015-08-07 17:51:44.567,2,view,325215,NaN


In [5]:
print(
    clean_events["event"]
    .value_counts()
)

event
view           2664218
addtocart        68966
transaction      22457
Name: count, dtype: int64


In [6]:
events_with_sessions = create_sessions(
    clean_events
)

print(
    "Number of sessions:",
    events_with_sessions["session_id"].nunique()
)

display(
    events_with_sessions[
        [
            "visitorid",
            "datetime",
            "event",
            "session_id",
        ]
    ].head(20)
)

Number of sessions: 1761675


,visitorid,datetime,event,session_id
0,0,2015-09-11 20:49:49.439,view,0_1
1,0,2015-09-11 20:52:39.591,view,0_1
2,0,2015-09-11 20:55:17.175,view,0_1
3,1,2015-08-13 17:46:06.444,view,1_1
4,2,2015-08-07 17:51:44.567,view,2_1
5,2,2015-08-07 17:53:33.790,view,2_1
6,2,2015-08-07 17:56:52.664,view,2_1
7,2,2015-08-07 18:01:08.920,view,2_1
8,2,2015-08-07 18:08:25.669,view,2_1
9,2,2015-08-07 18:17:24.375,view,2_1


In [7]:
model_events, labels = (
    create_labels_and_model_events(
        events_with_sessions
    )
)

print(
    "Valid modelling sessions:",
    labels["session_id"].nunique()
)

print("\nTarget distribution:")

print(
    labels["target"]
    .value_counts()
    .sort_index()
)

print("\nTarget percentages:")

print(
    labels["target"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(4)
)

Valid modelling sessions: 1760737

Target distribution:
target
0    1747378
1      13359
Name: count, dtype: int64

Target percentages:
target
0    99.2413
1     0.7587
Name: proportion, dtype: float64


In [8]:
assert set(
    model_events["event"].unique()
).issubset(ALLOWED_INPUT_EVENTS)

assert not (
    model_events["event"] == "transaction"
).any()

assert set(
    labels["target"].unique()
).issubset({0, 1})

assert (
    model_events["session_id"].nunique()
    == labels["session_id"].nunique()
)

print("Leakage checks passed.")

Leakage checks passed.


In [ ]:
split = chronological_split(
    model_events,
    labels,
    output_dir=processed_dir
)

print("Train sessions:", len(split["train"]))
print("Validation sessions:", len(split["val"]))
print("Test sessions:", len(split["test"]))

Train sessions: 1232515
Validation sessions: 264110
Test sessions: 264112


In [1]:
print("model_events exists:", "model_events" in globals())
print("labels exists:", "labels" in globals())
print("processed_dir exists:", "processed_dir" in globals())

model_events exists: False
labels exists: False
processed_dir exists: False


In [2]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

processed_dir = PROJECT_ROOT / "data" / "processed"

model_events = pd.read_parquet(
    processed_dir / "model_events.parquet"
)

labels = pd.read_csv(
    processed_dir / "labels.csv"
)

print("Model events loaded:", len(model_events))
print("Labels loaded:", len(labels))
print("Processed directory:", processed_dir)

Model events loaded: 2686671
Labels loaded: 1760737
Processed directory: d:\Ecommerce-purchase-prediction-DeepLearning\data\processed


In [3]:
print("Model events:", len(model_events))
print("Labels:", len(labels))
print("Processed directory:", processed_dir)

print("\nFirst few labels:")
display(labels.head())

Model events: 2686671
Labels: 1760737
Processed directory: d:\Ecommerce-purchase-prediction-DeepLearning\data\processed

First few labels:


,session_id,target
0,0_1,0
1,1000000_1,0
2,1000001_1,0
3,1000001_2,0
4,1000001_3,0


In [10]:
train_ids = set(split["train"])
val_ids = set(split["val"])
test_ids = set(split["test"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

all_split_ids = (
    train_ids
    | val_ids
    | test_ids
)

assert len(all_split_ids) == len(labels)

print("Split overlap checks passed.")

Split overlap checks passed.


In [11]:
for split_name, session_ids in split.items():

    subset = labels[
        labels["session_id"].isin(
            session_ids
        )
    ]

    print(f"\n{split_name.upper()}")

    print(
        subset["target"]
        .value_counts()
        .sort_index()
    )

    print(
        subset["target"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(4)
    )


TRAIN
target
0    1223084
1       9431
Name: count, dtype: int64
target
0    99.2348
1     0.7652
Name: proportion, dtype: float64

VAL
target
0    262119
1      1991
Name: count, dtype: int64
target
0    99.2461
1     0.7539
Name: proportion, dtype: float64

TEST
target
0    262175
1      1937
Name: count, dtype: int64
target
0    99.2666
1     0.7334
Name: proportion, dtype: float64


In [12]:
model_events_path = (
    processed_dir / "model_events.parquet"
)

labels_path = (
    processed_dir / "labels.csv"
)

model_events.to_parquet(
    model_events_path,
    index=False
)

labels.to_csv(
    labels_path,
    index=False
)

print("Saved:")
print(model_events_path)
print(labels_path)

Saved:
d:\Ecommerce-purchase-prediction-DeepLearning\data\processed\model_events.parquet
d:\Ecommerce-purchase-prediction-DeepLearning\data\processed\labels.csv


In [13]:
print("FINAL FOUNDATION SUMMARY")
print("-" * 40)

print(
    "Model event rows:",
    len(model_events)
)

print(
    "Sessions:",
    labels["session_id"].nunique()
)

print(
    "Positive sessions:",
    int(labels["target"].sum())
)

print(
    "Negative sessions:",
    int((labels["target"] == 0).sum())
)

print(
    "Train:",
    len(split["train"])
)

print(
    "Validation:",
    len(split["val"])
)

print(
    "Test:",
    len(split["test"])
)

print("\nFoundation pipeline completed successfully.")

FINAL FOUNDATION SUMMARY
----------------------------------------
Model event rows: 2686671
Sessions: 1760737
Positive sessions: 13359
Negative sessions: 1747378
Train: 1232515
Validation: 264110
Test: 264112

Foundation pipeline completed successfully.
